In [1]:
import os
import sys
import numpy as np
import pandas as pd
from tqdm import tqdm

sys.path.append(os.path.abspath("../../")) ; from EPF import variables
sys.path.insert(0, "../helpers/"); from run_parrellel import run_parallel

In [2]:
def train_valid_test_split(data):
    train = data[(data.index >= variables.TRAIN_START) & (data.index <= variables.VALID_START)]
    validate = data[(data.index > variables.VALID_START) & (data.index <= variables.TEST_START)]
    test  = data[data.index > variables.TEST_START]

    return train, validate, test

y_train, y_validate, y_test = train_valid_test_split(pd.read_parquet(variables.AGG_TARGET_DATASET_PATH))

In [3]:
display(y_train.shape)
display(y_validate.shape)
display(y_test.shape)

(578305, 96)

(52992, 96)

(105120, 96)

In [4]:
# 1D sample-weight array of shape (n_train_rows,) — not a feature, never enters X_train.
# Each entry scales how much that row contributes to the loss during LightGBM training.
# linspace(0, 1.5, n) spaces exponents evenly across rows (oldest → newest), so
# exp() produces weights rising from 1.0 (oldest row) to ~4.5 (newest row),
# assuming X_train and y_train are aligned chronologically for training rows.
# Use X_train length because the weights correspond to training row order.

training_range = pd.date_range(variables.PIPELINE_START_DATE,variables.VALID_START,freq=f'{variables.FEATURE_GRANULARITY_IN_MINUTES}min')

chronological_weights = np.exp(np.linspace(0.0, 1.5, len(training_range))).astype(np.float32)

In [5]:
print(chronological_weights.shape)

(578305,)


In [6]:
def _create_y_derivatives_per_target_col(horizon: int):
    """Compute all y (target) transformations for a given horizon."""
    
    target_col = f"target_h{horizon}"
    
    PRICE_TRANSFORM_SCALE = 100
    STANDARD_CLIP_PERCENTILE = 97.0
    POSITIVE_SPIKE_THRESHOLD = 150.0
    NEGATIVE_PRICE_THRESHOLD = 0.0
    POSITIVE_SPIKE_UPWEIGHT = 10.0
    NEGATIVE_SPIKE_UPWEIGHT = 7.0

    # y_train/y_validate hold all horizons as columns; slice the one being fitted
    # and convert to float32 to match LightGBM's expected dtype
    y_train_individual_target = y_train[target_col].values.astype(np.float32)
    y_validate_individual_target = y_validate[target_col].values.astype(np.float32)

    # Compute sample weights for standard model training
    # Returns a 1D sample-weight array of shape (n_train_rows,) combining two effects:
    #   - recency: weights already rise from 1.0 → ~4.5 via _chronological_weights
    #   - spike rows (price > threshold) are further multiplied by 3x
    # Net result: a recent spike row is weighted up to ~13.4x vs an old normal row,
    # preventing the standard model's loss from treating spikes as negligible noise.
    y_train_loss_weights = chronological_weights.astype(np.float32)
    y_train_full_range_loss_weights = (chronological_weights * np.where(y_train_individual_target > POSITIVE_SPIKE_THRESHOLD, 3.0, 1.0)).astype(np.float32)
    y_train_positive_spike_loss_weights = (chronological_weights * np.where(y_train_individual_target > POSITIVE_SPIKE_THRESHOLD, POSITIVE_SPIKE_UPWEIGHT, 1.0)).astype(np.float32)
    y_train_negative_spike_loss_weights = (chronological_weights * np.where(y_train_individual_target > NEGATIVE_PRICE_THRESHOLD, NEGATIVE_SPIKE_UPWEIGHT, 1.0)).astype(np.float32)

    # Compute clip threshold from training data
    y_train_individual_clipped = float(np.percentile(y_train_individual_target, STANDARD_CLIP_PERCENTILE))

    # arcsinh(y / 100) is a variance-stabilising transform that compresses large values
    # while preserving sign — unlike log, it handles negative prices. np.minimum clips
    # at the 97th-percentile threshold before transforming so extreme spikes don't
    # dominate the standard model's loss; the spike models handle those separately.
    y_train_individual_clipped_transformed = np.arcsinh(np.minimum(y_train_individual_target, y_train_individual_clipped) / PRICE_TRANSFORM_SCALE).astype(np.float32)
    y_validation_individual_clipped_transformed = np.arcsinh(np.minimum(y_validate_individual_target, y_train_individual_clipped) / PRICE_TRANSFORM_SCALE).astype(np.float32)

    # Same arcsinh transform but on the uncapped target — full price range including
    # extreme spikes and negative prices, which the specialist models are trained on.
    y_train_individual_unclipped_transformed = np.arcsinh(y_train_individual_target / PRICE_TRANSFORM_SCALE).astype(np.float32)
    y_validation_individual_unclipped_transformed = np.arcsinh(y_validate_individual_target / PRICE_TRANSFORM_SCALE).astype(np.float32)

    # Compute positive spike labels
    positive_spike_labels_train = (y_train_individual_target > POSITIVE_SPIKE_THRESHOLD).astype(np.float32)
    positive_spike_labels_validate = (y_validate_individual_target > POSITIVE_SPIKE_THRESHOLD).astype(np.float32)

    # Compute negative spike labels
    negative_spike_labels_train = (y_train_individual_target < NEGATIVE_PRICE_THRESHOLD).astype(np.float32)
    negative_spike_labels_validate = (y_validate_individual_target < NEGATIVE_PRICE_THRESHOLD).astype(np.float32)

    return {
        # 1 - target clipped and transformed
        'y_train_individual_clipped_transformed': y_train_individual_clipped_transformed,
        'y_validation_individual_clipped_transformed': y_validation_individual_clipped_transformed,

        # 2 - target no clipped but transformed
        'y_train_individual_unclipped_transformed': y_train_individual_unclipped_transformed,
        'y_validation_individual_unclipped_transformed': y_validation_individual_unclipped_transformed,
        
        # 3 - Positive target spike labels
        'positive_spike_labels_train': positive_spike_labels_train,
        'positive_spike_labels_validate': positive_spike_labels_validate,

        # 4 - Negative target spike labels
        'negative_spike_labels_train': negative_spike_labels_train,
        'negative_spike_labels_validate': negative_spike_labels_validate,

        # 5 - Weighting curve of target inc recency and spikes
        'y_train_loss_weights': y_train_loss_weights,
        'y_train_full_range_loss_weights': y_train_full_range_loss_weights,
        'y_train_positive_spike_loss_weights': y_train_positive_spike_loss_weights,
        'y_train_negative_spike_loss_weights': y_train_negative_spike_loss_weights
    }



horizon_list = list(range(1, variables.HORIZON_COUNT+1))

derived_targets = {h: _create_y_derivatives_per_target_col(h) for h in horizon_list}

target_dfs = {
    k: pd.DataFrame({f"h{h}": derived_targets[h][k] for h in horizon_list})
    for k in next(iter(derived_targets.values())).keys()
}

for df in target_dfs:
    target_dfs[df].to_parquet(f"Data/1_derived_targets/{df}.parquet")
